# Set up in cloud

### For Colab notebooks, start here

In [ ]:
!git clone https://${GITHUB_TOKEN}@github.com/getsentry/grouping-trainer.git

In [ ]:
%cd grouping-trainer

### For Workbench notebooks, start here

In [1]:
!git rev-parse --short HEAD

605ee76


After running this `pip install` cell, restart the notebook session. TODO: activate venv instead

In [2]:
!pip install -e .

Obtaining file:///home/jupyter/grouping-trainer
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.0/38.0 MB 133.4 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 127.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 165.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 888.0/888.0 MB 40.3 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 62.1 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 20.2 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 64.8 MB/s  0:00:016m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 2.3 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 5

In [1]:
!gsutil -m cp -r gs://seer-models/models/issue_grouping_v1 .

Copying gs://seer-models/models/issue_grouping_v1/.DS_Store...
Copying gs://seer-models/models/issue_grouping_v1/data.pkl...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/1_Pooling/config.json...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/config.json...     
Copying gs://seer-models/models/issue_grouping_v1/embeddings/README.md...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/config_sentence_transformers.json...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/merges.txt...      
Copying gs://seer-models/models/issue_grouping_v1/embeddings/configuration_bert.py...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/model.safetensors...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/modeling_bert.py...
Copying gs://seer-models/models/issue_grouping_v1/embeddings/modules.json...    
Copying gs://seer-models/models/issue_grouping_v1/embeddings/special_tokens_map.json...
Copying gs://seer-models/models/issue

In [6]:
!mkdir gte-finetuned
!gsutil -m cp -r gs://grouping-data/runs/./2025-12-19-15-53-06-gte-output/training gte-finetuned/

Copying gs://grouping-data/runs/./2025-12-19-15-53-06-gte-output/training/1_Pooling/config.json...
Copying gs://grouping-data/runs/./2025-12-19-15-53-06-gte-output/training/README.md...
Copying gs://grouping-data/runs/./2025-12-19-15-53-06-gte-output/training/checkpoint-3300/1_Pooling/config.json...
Copying gs://grouping-data/runs/./2025-12-19-15-53-06-gte-output/training/checkpoint-3300/README.md...
Copying gs://grouping-data/runs/./2025-12-19-15-53-06-gte-output/training/checkpoint-3300/config.json...
Copying gs://grouping-data/runs/./2025-12-19-15-53-06-gte-output/training/checkpoint-3300/config_sentence_transformers.json...
Copying gs://grouping-data/runs/./2025-12-19-15-53-06-gte-output/training/checkpoint-3300/model.safetensors...
Copying gs://grouping-data/runs/./2025-12-19-15-53-06-gte-output/training/checkpoint-3300/modules.json...
Copying gs://grouping-data/runs/./2025-12-19-15-53-06-gte-output/training/checkpoint-3300/optimizer.pt...
Copying gs://grouping-data/runs/./2025-12

In [3]:
!gsutil -m -o GSUtil:check_hashes=never cp -r gs://grouping-data/final_csvs .

Copying gs://grouping-data/final_csvs/sentry.csv...
Copying gs://grouping-data/final_csvs/synthetic-semi-easy-negatives.csv...      
Copying gs://grouping-data/final_csvs/test.csv...                               
Copying gs://grouping-data/final_csvs/train.csv...                              
Copying gs://grouping-data/final_csvs/train_no_sentry.csv...                    
Copying gs://grouping-data/final_csvs/val.csv...                                
- [6/7 files][  9.0 GiB/  9.0 GiB]  99% Done 140.2 MiB/s ETA 00:00:00           

# Run

In [14]:
import os
import json
from datetime import datetime

import polars as pl
from pydantic import BaseModel, field_serializer
from sentence_transformers.util import pairwise_cos_sim
import torch
from tqdm.auto import tqdm

import grouping_trainer as gt
import utils

In [ ]:
class ModelConfig(BaseModel):
    name: str
    path: str
    truncate_dim: int | None = None
    batch_size: int = 1
    model_kwargs: dict | None = None

    @field_serializer("model_kwargs")
    def serialize_model_kwargs(self, v: dict | None) -> dict:
        if v is None:
            return None
        return {k: str(val) if isinstance(val, torch.dtype) else val for k, val in v.items()}


class ModelConfigs(BaseModel):
    model_configs: list[ModelConfig]


class DataConfig(BaseModel):
    val_df_path: str
    sample_size: int | None = None

In [3]:
timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")

In [19]:
RUN_SHORTNAME = "test"
DATA_CONFIG = DataConfig(
    val_df_path="final_csvs/test.csv",
    sample_size=100,  # TODO: change
)
MODEL_CONFIGS = ModelConfigs(
    model_configs=[
        ModelConfig(
            name="prod",
            path="issue_grouping_v1/embeddings",
            truncate_dim=None,
        ),
        ModelConfig(
            name="gte-finetuned",
            path="gte-finetuned/training",
            truncate_dim=64,
            model_kwargs=dict(
                dtype=torch.bfloat16,
                attn_implementation="sdpa",
            ),
        ),
    ]
)

OUTPUT_DIR = f"./{timestamp}-{RUN_SHORTNAME}"

In [20]:
df = utils.load_val_df(path=DATA_CONFIG.val_df_path, sample_size=DATA_CONFIG.sample_size)

for model_config in tqdm(MODEL_CONFIGS.model_configs, desc="Processing models"):
    print(model_config)
    model = gt.utils.SentenceTransformer(
        model_config.path,
        trust_remote_code=True,
        truncate_dim=model_config.truncate_dim,
    )
    query_embeddings = model.encode(
        df["query_stacktrace_string"].to_list(),
        batch_size=model_config.batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
    )
    candidate_embeddings = model.encode(
        df["candidate_stacktrace_string"].to_list(),
        batch_size=model_config.batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
    )
    cos_sims = pairwise_cos_sim(query_embeddings, candidate_embeddings).detach().cpu().numpy()
    df = df.with_columns(pl.Series(name=f"cos_sim_{model_config.name}", values=cos_sims))
    print()

Processing models:   0%|          | 0/2 [00:00<?, ?it/s]

name='prod' path='issue_grouping_v1/embeddings' truncate_dim=None batch_size=1 model_kwargs=None


Batches:   0%|          | 0/99 [00:00<?, ?it/s]

Batches:   0%|          | 0/99 [00:00<?, ?it/s]


name='gte-finetuned' path='gte-finetuned/training' truncate_dim=64 batch_size=1 model_kwargs={'dtype': torch.bfloat16, 'attn_implementation': 'sdpa'}


Batches:   0%|          | 0/99 [00:00<?, ?it/s]

Batches:   0%|          | 0/99 [00:00<?, ?it/s]

In [21]:
print(df.columns)

['query_seer_event_sent', 'candidate_seer_event_sent', 'distance', 'query_group_id', 'candidate_group_id', 'query_hash', 'candidate_hash', 'query_grouphash_id', 'candidate_grouphash_id', 'query_grouphashmetadata_id', 'candidate_grouphashmetadata_id', 'query_seer_gr_id', 'candidate_seer_gr_id', 'query_error_type', 'candidate_error_type', 'project_id', 'platform', 'source', 'path', 'query_stacktrace_string', 'candidate_stacktrace_string', 'label', 'thinking_output', 'response_output', 'confidence_score', 'prompt', 'org_id', 'is_grouped', 'cos_sim_prod', 'cos_sim_gte-finetuned']


# Upload

In [11]:
os.mkdir(OUTPUT_DIR)

In [22]:
with open(f"{OUTPUT_DIR}/model_configs.json", "w") as f:
    json.dump(MODEL_CONFIGS.model_dump(), f, indent=4)

with open(f"{OUTPUT_DIR}/data_config.json", "w") as f:
    json.dump(DATA_CONFIG.model_dump(), f, indent=4)

In [23]:
df.write_csv(f"{OUTPUT_DIR}/similarities.csv")

In [24]:
!gsutil -m rsync -r {OUTPUT_DIR} gs://grouping-data/similarities/{OUTPUT_DIR}

Building synchronization state...
Starting synchronization...
Copying file://./2026-01-08-13-06-28-test/.ipynb_checkpoints/data_config-checkpoint.json [Content-Type=application/json]...
Copying file://./2026-01-08-13-06-28-test/data_config.json [Content-Type=application/json]...
Copying file://./2026-01-08-13-06-28-test/.ipynb_checkpoints/model_configs-checkpoint.json [Content-Type=application/json]...
Copying file://./2026-01-08-13-06-28-test/model_configs.json [Content-Type=application/json]...
Copying file://./2026-01-08-13-06-28-test/similarities.csv [Content-Type=text/csv]...
/ [5/5 files][761.4 KiB/761.4 KiB] 100% Done                                    
Operation completed over 5 objects/761.4 KiB.                                    


In [26]:
!gsutil -m cp -r run.ipynb gs://grouping-data/similarities/{OUTPUT_DIR}

Copying file://run.ipynb [Content-Type=application/octet-stream]...
/ [1/1 files][ 52.1 KiB/ 52.1 KiB] 100% Done                                    
Operation completed over 1 objects/52.1 KiB.                                     
